# Otto Mini - 5% CV Full Pipeline

This notebook clones `oyu-col/mini_recsys` and runs the complete OTTO candidate generation plus ranker pipeline on the local validation dataset. Use Kaggle GPU if available.

## Dataset

Attach `OTTO train and validation (extracted from train)` in the Kaggle Notebook sidebar. The dataset must contain `train.parquet`, `test.parquet`, and `test_labels.parquet`. The code below uses `exp001_dev`, so the upstream sampling logic keeps a 5% CV subset through `pp.sampling_ratio: 0.05`.

In [ ]:
%%time
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/oyu-col/mini_recsys.git"
REPO_DIR = Path("/kaggle/working/mini_recsys")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

subprocess.run([
    "python", "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt",
    "--upgrade-strategy", "only-if-needed"
], check=True)

In [ ]:
%%time
import os
import sys
from pathlib import Path

required = {"train.parquet", "test.parquet", "test_labels.parquet"}
matches = []
for path in Path("/kaggle/input").rglob("*"):
    if path.is_dir() and all((path / name).exists() for name in required):
        matches.append(path)

if not matches:
    raise FileNotFoundError(
        "Could not find a mounted CV dataset containing train.parquet, "
        "test.parquet, and test_labels.parquet under /kaggle/input."
    )

cv_data_dir = sorted(matches, key=lambda p: len(str(p)))[0]
os.environ["OTTO_CV_DATASET_DIR"] = str(cv_data_dir)
os.environ["PYTHONPATH"] = str(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print(f"Detected CV dataset: {cv_data_dir}")
print("Experiment: exp001_dev")
print("Sampling ratio: 0.05, from yaml/exp001_dev.yaml")

In [ ]:
import os
import subprocess
from pathlib import Path

EXP = "exp001_dev"
REPO_DIR = Path("/kaggle/working/mini_recsys")

def run(cmd: str):
    print(f"$ {cmd}", flush=True)
    subprocess.run(
        cmd,
        shell=True,
        check=True,
        cwd=REPO_DIR,
        env=os.environ.copy(),
    )

## 001. Preprocess

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/data_loader/main.py --exp {EXP}")

## 002. Candidate Generation

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/cand_generator/last_inter/main.py --exp {EXP}")

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/cand_generator/item_cf/main.py --exp {EXP}")

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/cand_generator/item_mf/main.py --exp {EXP}")

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/cand_generator/user_mf/main.py --exp {EXP}")

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/cand_generator/item2vec/main.py --exp {EXP}")

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/cand_merger/main.py --exp {EXP}")

## 003. Feature Engineering

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/feature/main.py --exp {EXP}")

## 004. Train Ranker and CV Score

In [ ]:
%%time
run(f"PYTHONPATH=. python kaggle_otto2/ranker_trainer/main.py --exp {EXP} --model_type lgbm")

In [ ]:
from pathlib import Path

out_dir = Path("/kaggle/working/output") / EXP
print(f"Output directory: {out_dir}")
for path in sorted(out_dir.glob("*.txt"))[:20]:
    print(path)